# 02 | Dataset preparation and evaluation design

## Bainite–Martensite microscopy classification

This notebook constructs the cleaned image manifest and reproducible evaluation partitions used throughout the study. Image records are reconciled with acquisition metadata, exact duplicate files are identified, and phase, specimen, microscope and magnification distributions are examined before splitting.

**Methodological objective.** Separate conventional image-level classification from evaluation on unseen physical specimens and acquisition domains. The distinction is necessary because correlated images can yield optimistic performance estimates when their shared origin is not accounted for.

Image resizing, augmentation, normalization and model training are outside the scope of this preparation step.

## 1. Data locations and reproducibility

Define the raw-image directory, metadata CSV, output locations and random seed. Paths are relative to the notebook's project layout.

In [1]:
from pathlib import Path

IMAGE_DIR = Path(r"../data/raw/Dataset")
METADATA_PATH = Path(r"../data/raw/Dataset_Patches_Microscope_Metadata.csv")
OUTPUT_DIR = Path(r"../data/processed")
REPORT_DIR = Path(r"../results/02_dataset_preparation")
RANDOM_SEED = 42


## 2. Dependencies and input validation

Import tabular analysis and splitting utilities, create output directories and verify access to the source files.

In [2]:
import hashlib, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from IPython.display import display

warnings.filterwarnings("ignore")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Images   :", IMAGE_DIR.resolve())
print("Metadata :", METADATA_PATH.resolve())
print("Processed:", OUTPUT_DIR.resolve())

assert IMAGE_DIR.exists(), f"IMAGE_DIR not found: {IMAGE_DIR.resolve()}"
assert METADATA_PATH.exists(), f"METADATA_PATH not found: {METADATA_PATH.resolve()}"
print("\n✓ Paths validated.")


Images   : D:\AI_Materials_Microscopy\data\raw\Dataset
Metadata : D:\AI_Materials_Microscopy\data\raw\Dataset_Patches_Microscope_Metadata.csv
Processed: D:\AI_Materials_Microscopy\data\processed

✓ Paths validated.


## 3. Metadata integrity

Load the metadata and verify that the phase label and physical-specimen, microscope and magnification identifiers are available.

In [3]:
metadata = pd.read_csv(METADATA_PATH).copy()
metadata["metadata_row"] = np.arange(len(metadata))

required = ["Phase", "MicroscopeID", "Magnification", "SampleIDHarm"]
missing = [c for c in required if c not in metadata.columns]
assert not missing, f"Required columns missing: {missing}"

print("Metadata shape:", metadata.shape)
display(metadata.head())
display(metadata["Phase"].value_counts(dropna=False).to_frame("count"))


Metadata shape: (5327, 14)


,FileNameHarm,ID,SampleIDHarm,Magnification,ImageSizePx,MicroscopeID,EtchingID,ExposureHarm,ApertureHarm,ObjectivHarm,FilterHarm,Phase,PatchID,metadata_row
0,38_A_Bainite_01,38,A,20x,96,0,0,medium,high,standard,DLF,Bainite,1,0
1,38_A_Bainite_02,38,A,20x,96,0,0,medium,high,standard,DLF,Bainite,2,1
2,38_A_Bainite_03,38,A,20x,96,0,0,medium,high,standard,DLF,Bainite,3,2
3,38_A_Bainite_04,38,A,20x,96,0,0,medium,high,standard,DLF,Bainite,4,3
4,38_A_Bainite_05,38,A,20x,96,0,0,medium,high,standard,DLF,Bainite,5,4


,count
Phase,
Martensite,2850
Bainite,2477


## 4. Recursive image inventory

Enumerate supported image formats across all nested dataset directories. Record absolute and relative paths, filenames and folder identifiers for subsequent joins.

In [4]:
IMAGE_EXTENSIONS = {".png",".jpg",".jpeg",".tif",".tiff",".bmp",".webp"}
image_files = sorted(p for p in IMAGE_DIR.rglob("*")
                     if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

images = pd.DataFrame({
    "image_path":[str(p.resolve()) for p in image_files],
    "relative_path":[str(p.relative_to(IMAGE_DIR)) for p in image_files],
    "filename":[p.name for p in image_files],
    "stem":[p.stem for p in image_files],
    "parent_folder":[p.parent.name for p in image_files],
})
print(f"Metadata records: {len(metadata):,}")
print(f"Images discovered: {len(images):,}")
display(images.head())


Metadata records: 5,327
Images discovered: 5,328


,image_path,relative_path,filename,stem,parent_folder
0,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\38_A_Bainite_01.png,38_A_Bainite_01.png,38_A_Bainite_01,A_Bainite_1
1,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\39_A_Bainite_01.png,39_A_Bainite_01.png,39_A_Bainite_01,A_Bainite_1
2,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\40_A_Bainite_01.png,40_A_Bainite_01.png,40_A_Bainite_01,A_Bainite_1
3,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\41_A_Bainite_01.png,41_A_Bainite_01.png,41_A_Bainite_01,A_Bainite_1
4,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\42_A_Bainite_01.png,42_A_Bainite_01.png,42_A_Bainite_01,A_Bainite_1


## 5. Image–metadata correspondence

Mapping is established using observed identifiers rather than assuming that metadata-row order matches image-file order. Candidate filename, stem and relative-path relationships are evaluated first; a constrained `PatchID`-based fallback is available if direct matching is insufficient. Unresolved mappings are retained as explicit audit findings rather than assigned by assumption.

In [5]:
diagnostics=[]
for col in metadata.columns:
    vals=metadata[col].dropna().astype(str).str.strip()
    if vals.empty: continue
    for key in ["filename","stem","relative_path"]:
        ivals=images[key].astype(str).str.strip()
        if key=="relative_path":
            vals2=vals.str.replace("\\","/",regex=False)
            iset=set(ivals.str.replace("\\","/",regex=False))
        else:
            vals2=vals
            iset=set(ivals)
        rate=vals2.isin(iset).mean()
        diagnostics.append({"metadata_column":col,"image_key":key,
                            "matched_%":100*rate})
diagnostics=pd.DataFrame(diagnostics).sort_values("matched_%",ascending=False)
display(diagnostics.head(15))

print("\nUseful metadata examples:")
for col in ["FileNameHarm","PatchID","SampleIDHarm","Phase","MicroscopeID","Magnification"]:
    if col in metadata.columns:
        print(f"\n{col}: {metadata[col].astype(str).head(10).tolist()}")
print("\nRepresentative image names:")
display(images.head(20))


,metadata_column,image_key,matched_%
1,FileNameHarm,stem,99.981228
0,FileNameHarm,filename,0.000000
31,FilterHarm,stem,0.000000
23,ExposureHarm,relative_path,0.000000
24,ApertureHarm,filename,0.000000
25,ApertureHarm,stem,0.000000
26,ApertureHarm,relative_path,0.000000
27,ObjectivHarm,filename,0.000000
28,ObjectivHarm,stem,0.000000
29,ObjectivHarm,relative_path,0.000000



Useful metadata examples:

FileNameHarm: ['38_A_Bainite_01', '38_A_Bainite_02', '38_A_Bainite_03', '38_A_Bainite_04', '38_A_Bainite_05', '38_A_Bainite_06', '38_A_Martensite_01', '38_A_Martensite_02', '38_A_Martensite_03', '39_A_Bainite_01']

PatchID: ['1', '2', '3', '4', '5', '6', '1', '2', '3', '1']

SampleIDHarm: ['A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A']

Phase: ['Bainite', 'Bainite', 'Bainite', 'Bainite', 'Bainite', 'Bainite', 'Martensite', 'Martensite', 'Martensite', 'Bainite']

MicroscopeID: ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0']

Magnification: ['20x', '20x', '20x', '20x', '20x', '20x', '20x', '20x', '20x', '20x']

Representative image names:


,image_path,relative_path,filename,stem,parent_folder
0,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\38_A_Bainite_01.png,38_A_Bainite_01.png,38_A_Bainite_01,A_Bainite_1
1,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\39_A_Bainite_01.png,39_A_Bainite_01.png,39_A_Bainite_01,A_Bainite_1
2,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\40_A_Bainite_01.png,40_A_Bainite_01.png,40_A_Bainite_01,A_Bainite_1
3,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\41_A_Bainite_01.png,41_A_Bainite_01.png,41_A_Bainite_01,A_Bainite_1
4,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\42_A_Bainite_01.png,42_A_Bainite_01.png,42_A_Bainite_01,A_Bainite_1
5,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\43_A_Bainite_01.png,43_A_Bainite_01.png,43_A_Bainite_01,A_Bainite_1
6,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\44_A_Bainite_01.png,44_A_Bainite_01.png,44_A_Bainite_01,A_Bainite_1
7,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\45_A_Bainite_01.png,45_A_Bainite_01.png,45_A_Bainite_01,A_Bainite_1
8,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\46_A_Bainite_01.png,46_A_Bainite_01.png,46_A_Bainite_01,A_Bainite_1
9,D:\AI_Materials_Microscopy\data\raw\Dataset\A_...,A_Bainite_1\47_A_Bainite_01.png,47_A_Bainite_01.png,47_A_Bainite_01,A_Bainite_1


### 5.1 Construct the metadata mapping

Select the supported mapping key and build the image–metadata correspondence. The mapping method and match rate are reported for traceability.

In [6]:
best=diagnostics.iloc[0]
mapping=None
mapping_method=None

if best["matched_%"] >= 99:
    col,key=best["metadata_column"],best["image_key"]
    left=metadata.copy(); right=images.copy()
    left["_map_key"]=left[col].astype(str).str.strip()
    right["_map_key"]=right[key].astype(str).str.strip()
    if key=="relative_path":
        left["_map_key"]=left["_map_key"].str.replace("\\","/",regex=False)
        right["_map_key"]=right["_map_key"].str.replace("\\","/",regex=False)
    mapping=left.merge(right,on="_map_key",how="left")
    mapping_method=f"{col} ↔ {key}"

if mapping is None and "PatchID" in metadata.columns:
    # Conservative fallback: a unique PatchID token must identify exactly one image.
    patch_series=metadata["PatchID"].dropna().astype(str).str.strip()
    unique_patches=set(patch_series[~patch_series.duplicated(keep=False)])
    lookup={}
    for patch in unique_patches:
        pat=re.compile(rf"(^|[^A-Za-z0-9]){re.escape(patch)}([^A-Za-z0-9]|$)",re.I)
        hits=images[images["stem"].astype(str).apply(lambda x:bool(pat.search(x)))]
        if len(hits)==1:
            lookup[patch]=hits.iloc[0]["image_path"]
    temp=metadata.copy()
    temp["image_path"]=temp["PatchID"].astype(str).str.strip().map(lookup)
    rate=temp["image_path"].notna().mean()
    print(f"PatchID fallback coverage: {rate:.2%}")
    if rate>=.99:
        mapping=temp.merge(images,on="image_path",how="left")
        mapping_method="unique PatchID token in filename"

assert mapping is not None, (
    "Reliable image↔metadata mapping was not established. "
    "Do not bypass this safeguard; inspect the diagnostics above."
)
print("Mapping method:",mapping_method)
print(f"Mapped: {mapping['image_path'].notna().sum():,}/{len(metadata):,}")


Mapping method: FileNameHarm ↔ stem
Mapped: 5,326/5,327


### 5.2 Unmatched records and mapping multiplicity

Report metadata rows without corresponding images, images without corresponding metadata and any image path associated with multiple metadata rows. These discrepancies must be distinguished from duplicated image content.

In [7]:
matched_paths=set(mapping["image_path"].dropna())
unmatched_metadata=mapping[mapping["image_path"].isna()].copy()
unmatched_images=images[~images["image_path"].isin(matched_paths)].copy()

print("Unmatched metadata rows:",len(unmatched_metadata))
print("Unmatched image files  :",len(unmatched_images))
if len(unmatched_metadata): display(unmatched_metadata)
if len(unmatched_images): display(unmatched_images)

counts=mapping["image_path"].value_counts(dropna=True)
print("Images mapped to >1 metadata row:",(counts>1).sum())


Unmatched metadata rows: 1
Unmatched image files  : 2


,FileNameHarm,ID,SampleIDHarm,Magnification,ImageSizePx,MicroscopeID,EtchingID,ExposureHarm,ApertureHarm,ObjectivHarm,FilterHarm,Phase,PatchID,metadata_row,_map_key,image_path,relative_path,filename,stem,parent_folder
1102,228_D_Bainite_01,228,D,20x,96,0,0,medium,high,standard,DLF,Bainite,1,1102,228_D_Bainite_01,NaN,NaN,NaN,NaN,NaN


,image_path,relative_path,filename,stem,parent_folder
1102,D:\AI_Materials_Microscopy\data\raw\Dataset\D_...,D_Bainite_1\228_C_Bainite_01.png,228_C_Bainite_01.png,228_C_Bainite_01,D_Bainite_1
5118,D:\AI_Materials_Microscopy\data\raw\Dataset\T_...,T_Bainite_1\.ipynb_checkpoints\886_T_Bainite_0...,886_T_Bainite_01-checkpoint.png,886_T_Bainite_01-checkpoint,.ipynb_checkpoints


Images mapped to >1 metadata row: 0


## 6. Exact duplicate audit

MD5 hashes identify files with identical byte content. Duplicate groups are checked for conflicting `Phase` labels before redundant records are removed.

This procedure detects **exact file duplicates**, not near-duplicates or images of adjacent regions from the same physical specimen. Specimen-disjoint evaluation therefore remains necessary even after hash deduplication.

In [8]:
def md5_file(path,chunk=1024*1024):
    h=hashlib.md5()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

hash_lookup={p:md5_file(Path(p)) for p in mapping["image_path"].dropna().unique()}
mapping["image_hash"]=mapping["image_path"].map(hash_lookup)

dups=mapping[mapping["image_hash"].duplicated(False) & mapping["image_hash"].notna()].copy()
print("Duplicate hash groups:",dups["image_hash"].nunique())
print("Records involved:",len(dups))

consistency=[]
for h,g in dups.groupby("image_hash"):
    consistency.append({
        "image_hash":h,"n_records":len(g),
        "n_phase":g["Phase"].nunique(dropna=False),
        "phases":" | ".join(sorted(g["Phase"].astype(str).unique())),
        "n_samples":g["SampleIDHarm"].nunique(dropna=False),
        "n_microscopes":g["MicroscopeID"].nunique(dropna=False),
        "n_magnifications":g["Magnification"].nunique(dropna=False),
    })
consistency=pd.DataFrame(consistency)
if len(consistency): display(consistency)
assert len(consistency)==0 or (consistency["n_phase"]<=1).all(), (
    "Conflicting Phase labels exist among identical images. Investigate first."
)

clean=mapping[mapping["image_path"].notna()].copy()
clean=clean.sort_values(["image_hash","image_path","metadata_row"])
clean["is_exact_duplicate"]=clean.duplicated("image_hash",keep="first")
removed_duplicates=clean[clean["is_exact_duplicate"]].copy()
clean=clean[~clean["is_exact_duplicate"]].copy().reset_index(drop=True)

print("Duplicate records removed:",len(removed_duplicates))
print("Unique modelling images :",len(clean))


Duplicate hash groups: 33
Records involved: 66


,image_hash,n_records,n_phase,phases,n_samples,n_microscopes,n_magnifications
0,0d2684af56c86660a1c70142b9437388,2,1,Martensite,1,1,1
1,0db71fb5af715780f7577425fd58951d,2,1,Martensite,1,1,1
2,1003e2cf00b411e2ce0902b3f9d35cfb,2,1,Bainite,1,1,1
3,152235f7c421bfebe98c660353c4a758,2,1,Martensite,1,1,1
4,2c262c49147c11cd9adef04675dc94bd,2,1,Martensite,1,1,1
5,416a5020ec4b385b5642c5d3ea70dd43,2,1,Martensite,1,1,1
6,4499ff55107b033b1cd559bd59b302de,2,1,Bainite,1,1,1
7,45eee139914c3f2c28676b962b64a259,2,1,Martensite,1,1,1
8,48f5c6b51126919777311d4177326093,2,1,Bainite,1,1,1
9,4f11c5bac580cf2d8f16e10f07d1c8c3,2,1,Martensite,1,1,1


Duplicate records removed: 33
Unique modelling images : 5293


## 7. Structure of the cleaned image collection

Cross-tabulate phase against microscope, magnification and physical-specimen identifiers. These summaries characterize confounding, domain coverage and class imbalance relevant to subsequent holdout experiments.

In [9]:
print("PHASE × MICROSCOPE")
display(pd.crosstab(clean["MicroscopeID"],clean["Phase"],margins=True))

print("\nPHASE × MAGNIFICATION")
display(pd.crosstab(clean["Magnification"],clean["Phase"],margins=True))

print("\nPHASE × SAMPLE")
display(pd.crosstab(clean["SampleIDHarm"],clean["Phase"],margins=True))

print("\nMICROSCOPE × MAGNIFICATION × PHASE")
display(pd.crosstab([clean["MicroscopeID"],clean["Magnification"]],
                    clean["Phase"],margins=True))

print("\nSAMPLE × MICROSCOPE")
display(pd.crosstab(clean["SampleIDHarm"],clean["MicroscopeID"]))


PHASE × MICROSCOPE


Phase,Bainite,Martensite,All
MicroscopeID,,,
0,1047,1200,2247
1,393,450,843
2,1030,1173,2203
All,2470,2823,5293



PHASE × MAGNIFICATION


Phase,Bainite,Martensite,All
Magnification,,,
100x,127,120,247
20x,1178,1338,2516
50x,1165,1365,2530
All,2470,2823,5293



PHASE × SAMPLE


Phase,Bainite,Martensite,All
SampleIDHarm,,,
A,222,111,333
B,114,190,304
C,0,444,444
D,683,0,683
E,0,216,216
F,0,228,228
G,0,228,228
H,0,190,190
I,0,228,228



MICROSCOPE × MAGNIFICATION × PHASE


Phase                       Bainite  Martensite   All
MicroscopeID Magnification                           
0            20x                523         600  1123
             50x                524         600  1124
1            100x               127         120   247
             20x                131         150   281
             50x                135         180   315
2            20x                524         588  1112
             50x                506         585  1091
All                            2470        2823  5293


SAMPLE × MICROSCOPE


MicroscopeID,0,1,2
SampleIDHarm,,,
A,144,54,135
B,128,48,128
C,192,72,180
D,287,108,288
E,96,36,84
F,96,36,96
G,96,36,96
H,80,30,80
I,96,36,96


## 8. Final modelling manifest

Encode `Bainite = 0` and `Martensite = 1` and retain the image path, phase label, source identifiers and acquisition metadata required for training and evaluation.

In [10]:
LABEL_MAP={"Bainite":0,"Martensite":1}
unexpected=set(clean["Phase"].dropna().unique())-set(LABEL_MAP)
assert not unexpected,f"Unexpected phases: {unexpected}"
clean["label"]=clean["Phase"].map(LABEL_MAP).astype(int)

preferred=["image_path","relative_path","filename","image_hash","Phase","label",
           "SampleIDHarm","MicroscopeID","Magnification","PatchID","FileNameHarm",
           "ImageSizePx","metadata_row"]
manifest=clean[[c for c in preferred if c in clean.columns]].copy()

print("Manifest:",manifest.shape)
display(manifest.head())


Manifest: (5293, 13)


,image_path,relative_path,filename,image_hash,Phase,label,SampleIDHarm,MicroscopeID,Magnification,PatchID,FileNameHarm,ImageSizePx,metadata_row
0,D:\AI_Materials_Microscopy\data\raw\Dataset\G_...,G_Martensite_1\412_G_Martensite_01.png,412_G_Martensite_01.png,000007c820efe4b46f2983fb2ae2af94,Martensite,1,G,1,50x,1,412_G_Martensite_01,256,2398
1,D:\AI_Materials_Microscopy\data\raw\Dataset\B_...,B_Martensite_1\93_B_Martensite_01.png,93_B_Martensite_01.png,0001369954e46828ef57bc6d27ac1ab7,Martensite,1,B,2,20x,1,93_B_Martensite_01,96,481
2,D:\AI_Materials_Microscopy\data\raw\Dataset\K_...,K_Bainite_4\559_K_Bainite_04.png,559_K_Bainite_04.png,000281d14b1ab4a7df1bfa5c481ad31e,Bainite,0,K,2,20x,4,559_K_Bainite_04,96,3215
3,D:\AI_Materials_Microscopy\data\raw\Dataset\D_...,D_Bainite_11\267_D_Bainite_11.png,267_D_Bainite_11.png,00066bdd25cc41118a9ac01becb9ff4f,Bainite,0,D,0,50x,11,267_D_Bainite_11,256,1580
4,D:\AI_Materials_Microscopy\data\raw\Dataset\F_...,F_Martensite_6\390_F_Martensite_06.png,390_F_Martensite_06.png,000a3975a545d5d2c5b5594334d0e5e6,Martensite,1,F,0,50x,6,390_F_Martensite_06,256,2139


## 9. Conventional stratified partition

A 70/15/15 train/validation/test split is generated with phase stratification. Class proportions are preserved, but images originating from the same physical specimen may occur in multiple partitions.

This provides an **image-level benchmark**, not independent evidence of transfer to unseen physical specimens.

In [11]:
tr,tmp=train_test_split(manifest,test_size=.30,stratify=manifest["label"],
                        random_state=RANDOM_SEED)
va,te=train_test_split(tmp,test_size=.50,stratify=tmp["label"],
                       random_state=RANDOM_SEED)

baseline=manifest.copy()
baseline["split"]="unassigned"
baseline.loc[baseline.index.isin(tr.index),"split"]="train"
baseline.loc[baseline.index.isin(va.index),"split"]="val"
baseline.loc[baseline.index.isin(te.index),"split"]="test"
display(pd.crosstab(baseline["split"],baseline["Phase"],margins=True))

def ov(a,b,col): return sorted(set(a[col])&set(b[col]))
print("Sample overlap train/test:",ov(tr,te,"SampleIDHarm"))
print("Hash overlap train/test:",len(ov(tr,te,"image_hash")))


Phase,Bainite,Martensite,All
split,,,
test,371,423,794
train,1729,1976,3705
val,370,424,794
All,2470,2823,5293


Sample overlap train/test: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T']
Hash overlap train/test: 0


## 10. Physical-specimen-disjoint partition

A grouped split assigns each `SampleIDHarm` to exactly one of train, validation or test. Candidate assignments are evaluated for approximate image-count proportions and representation of both phases.

This protocol estimates performance on **held-out physical specimens** within the dataset, reducing dependence between training and evaluation images from a shared specimen.

In [12]:
def score_split(a,b,c):
    parts={"train":a,"val":b,"test":c}; desired={"train":.70,"val":.15,"test":.15}
    size=sum(abs(len(d)/len(manifest)-desired[n]) for n,d in parts.items())
    gp=manifest["label"].mean(); cls=0
    for d in parts.values():
        if d["label"].nunique()<2:return np.inf
        cls+=abs(d["label"].mean()-gp)
    return size+cls

best_score=np.inf; best_split=None
for seed in range(2000):
    g1=GroupShuffleSplit(n_splits=1,train_size=.70,random_state=seed)
    i,j=next(g1.split(manifest,groups=manifest["SampleIDHarm"]))
    a=manifest.iloc[i]; rest=manifest.iloc[j]
    g2=GroupShuffleSplit(n_splits=1,train_size=.50,random_state=seed+10000)
    k,l=next(g2.split(rest,groups=rest["SampleIDHarm"]))
    b,c=rest.iloc[k],rest.iloc[l]
    s=score_split(a,b,c)
    if s<best_score:
        best_score=s; best_split=(a.copy(),b.copy(),c.copy(),seed)

gtr,gva,gte,seed=best_split
assert not ov(gtr,gva,"SampleIDHarm")
assert not ov(gtr,gte,"SampleIDHarm")
assert not ov(gva,gte,"SampleIDHarm")
assert not ov(gtr,gte,"image_hash")

grouped=manifest.copy(); grouped["split"]="unassigned"
grouped.loc[grouped.index.isin(gtr.index),"split"]="train"
grouped.loc[grouped.index.isin(gva.index),"split"]="val"
grouped.loc[grouped.index.isin(gte.index),"split"]="test"

print("Selected seed:",seed," score:",round(best_score,4))
display(pd.crosstab(grouped["split"],grouped["Phase"],margins=True))
for s in ["train","val","test"]:
    vals=sorted(grouped.loc[grouped["split"]==s,"SampleIDHarm"].unique())
    print(f"{s:5s}: {vals}")


Selected seed: 1173  score: 0.0657


Phase,Bainite,Martensite,All
split,,,
test,342,418,760
train,1786,2025,3811
val,342,380,722
All,2470,2823,5293


train: ['A', 'C', 'D', 'E', 'G', 'H', 'J', 'M', 'N', 'P', 'Q', 'R', 'S', 'T']
val  : ['I', 'L', 'O']
test : ['B', 'F', 'K']


## 11. Leave-one-microscope-out evaluation

Each microscope is excluded from training and validation and used as the test acquisition domain. The remaining data form the development partitions, with physical-specimen grouping attempted for the train–validation split.

This is an **instrument holdout**; physical specimens may still have images obtained with other microscopes in the development data. It should not be interpreted as an independent specimen-and-instrument holdout.

In [13]:
microscope_manifests={}
microscope_summary=[]

for held in sorted(manifest["MicroscopeID"].unique()):
    test=manifest[manifest["MicroscopeID"]==held].copy()
    dev=manifest[manifest["MicroscopeID"]!=held].copy()
    chosen=None
    for seed in range(1000):
        gs=GroupShuffleSplit(n_splits=1,test_size=.18,random_state=seed)
        i,j=next(gs.split(dev,groups=dev["SampleIDHarm"]))
        a,b=dev.iloc[i],dev.iloc[j]
        if a["label"].nunique()==2 and b["label"].nunique()==2:
            chosen=(a.copy(),b.copy());break
    if chosen is None:
        a,b=train_test_split(dev,test_size=.18,stratify=dev["label"],
                             random_state=RANDOM_SEED)
    else:a,b=chosen

    exp=manifest.copy();exp["split"]="excluded"
    exp.loc[exp.index.isin(a.index),"split"]="train"
    exp.loc[exp.index.isin(b.index),"split"]="val"
    exp.loc[exp.index.isin(test.index),"split"]="test"
    microscope_manifests[str(held)]=exp

    for sn,d in [("train",a),("val",b),("test",test)]:
        microscope_summary.append({
            "heldout_microscope":held,"split":sn,"n_images":len(d),
            "n_samples":d["SampleIDHarm"].nunique(),
            "bainite":(d["Phase"]=="Bainite").sum(),
            "martensite":(d["Phase"]=="Martensite").sum()
        })

microscope_summary=pd.DataFrame(microscope_summary)
display(microscope_summary)


,heldout_microscope,split,n_images,n_samples,bainite,martensite
0,0,train,2497,16,1138,1359
1,0,val,549,4,285,264
2,0,test,2247,20,1047,1200
3,1,train,3646,16,1657,1989
4,1,val,804,4,420,384
5,1,test,843,20,393,450
6,2,train,2529,16,1143,1386
7,2,val,561,4,297,264
8,2,test,2203,20,1030,1173


## 12. Leave-one-magnification-out evaluation

Each magnification is separately assigned as the held-out test domain. Training and validation use images acquired at the other magnifications.

Performance should be interpreted alongside domain sample size and class composition. In particular, the 100× subset contains substantially fewer images than the 20× and 50× subsets.

In [14]:
magnification_manifests={}
mag_summary=[]

for held in sorted(manifest["Magnification"].astype(str).unique()):
    test=manifest[manifest["Magnification"].astype(str)==held].copy()
    dev=manifest[manifest["Magnification"].astype(str)!=held].copy()
    chosen=None
    for seed in range(1000):
        gs=GroupShuffleSplit(n_splits=1,test_size=.18,random_state=seed)
        i,j=next(gs.split(dev,groups=dev["SampleIDHarm"]))
        a,b=dev.iloc[i],dev.iloc[j]
        if a["label"].nunique()==2 and b["label"].nunique()==2:
            chosen=(a.copy(),b.copy());break
    if chosen is None:
        a,b=train_test_split(dev,test_size=.18,stratify=dev["label"],
                             random_state=RANDOM_SEED)
    else:a,b=chosen

    exp=manifest.copy();exp["split"]="excluded"
    exp.loc[exp.index.isin(a.index),"split"]="train"
    exp.loc[exp.index.isin(b.index),"split"]="val"
    exp.loc[exp.index.isin(test.index),"split"]="test"
    magnification_manifests[held]=exp

    for sn,d in [("train",a),("val",b),("test",test)]:
        mag_summary.append({
            "heldout_magnification":held,"split":sn,"n_images":len(d),
            "n_samples":d["SampleIDHarm"].nunique(),
            "bainite":(d["Phase"]=="Bainite").sum(),
            "martensite":(d["Phase"]=="Martensite").sum()
        })

mag_summary=pd.DataFrame(mag_summary)
display(mag_summary)


,heldout_magnification,split,n_images,n_samples,bainite,martensite
0,100x,train,4128,16,1869,2259
1,100x,val,918,4,474,444
2,100x,test,247,17,127,120
3,20x,train,2279,16,1034,1245
4,20x,val,498,4,258,240
5,20x,test,2516,20,1178,1338
6,50x,train,2265,16,1035,1230
7,50x,val,498,4,270,228
8,50x,test,2530,20,1165,1365


## 13. Integrity verification and manifest export

Check image-path existence, class encoding, uniqueness of retained image hashes and completeness of split assignments. Export the cleaned manifest, conventional and specimen-grouped partitions, and domain-holdout CSV files.

In [15]:
assert all(Path(p).exists() for p in manifest["image_path"])
assert set(manifest["label"].unique())=={0,1}
assert manifest["image_hash"].is_unique
assert set(baseline["split"].unique())=={"train","val","test"}
assert set(grouped["split"].unique())=={"train","val","test"}

manifest.to_csv(OUTPUT_DIR/"clean_manifest.csv",index=False)
baseline.to_csv(OUTPUT_DIR/"split_baseline_stratified.csv",index=False)
grouped.to_csv(OUTPUT_DIR/"split_sample_grouped.csv",index=False)

for held,exp in microscope_manifests.items():
    safe=re.sub(r"[^A-Za-z0-9_-]+","_",str(held))
    exp.to_csv(OUTPUT_DIR/f"split_holdout_microscope_{safe}.csv",index=False)

for held,exp in magnification_manifests.items():
    safe=re.sub(r"[^A-Za-z0-9_-]+","_",str(held))
    exp.to_csv(OUTPUT_DIR/f"split_holdout_magnification_{safe}.csv",index=False)

microscope_summary.to_csv(REPORT_DIR/"microscope_holdout_summary.csv",index=False)
mag_summary.to_csv(REPORT_DIR/"magnification_holdout_summary.csv",index=False)
if len(unmatched_images):unmatched_images.to_csv(REPORT_DIR/"unmatched_images.csv",index=False)
if len(unmatched_metadata):unmatched_metadata.to_csv(REPORT_DIR/"unmatched_metadata.csv",index=False)
if len(dups):dups.to_csv(REPORT_DIR/"exact_duplicate_records.csv",index=False)

print("✓ Integrity checks passed.")
print("\nGenerated manifests:")
for p in sorted(OUTPUT_DIR.glob("*.csv")):print(" •",p.name)


✓ Integrity checks passed.

Generated manifests:
 • clean_manifest.csv
 • split_baseline_stratified.csv
 • split_holdout_magnification_100x.csv
 • split_holdout_magnification_20x.csv
 • split_holdout_magnification_50x.csv
 • split_holdout_microscope_0.csv
 • split_holdout_microscope_1.csv
 • split_holdout_microscope_2.csv
 • split_sample_grouped.csv


## 14. Preparation summary

Report the record reconciliation, exact-duplicate removal, final modelling cohort, class composition and evaluation-partition sizes.

In [16]:
print("="*76)
print("DATASET PREPARATION SUMMARY")
print("="*76)
print(f"Raw metadata rows         : {len(metadata):,}")
print(f"Raw images                : {len(images):,}")
print(f"Mapped metadata rows      : {mapping['image_path'].notna().sum():,}")
print(f"Unmatched metadata        : {len(unmatched_metadata):,}")
print(f"Unmatched images          : {len(unmatched_images):,}")
print(f"Duplicate hash groups     : {dups['image_hash'].nunique():,}")
print(f"Duplicate records removed : {len(removed_duplicates):,}")
print(f"Final modelling images    : {len(manifest):,}")

print("\nTARGET")
print(manifest["Phase"].value_counts().to_string())

print("\nSAMPLE-GROUPED SPLIT")
print(pd.crosstab(grouped["split"],grouped["Phase"]).to_string())

print("\nMICROSCOPE HOLDOUTS")
print(microscope_summary.to_string(index=False))

print("\nMAGNIFICATION HOLDOUTS")
print(mag_summary.to_string(index=False))

print("\nNEXT: 03_baseline_cnn.ipynb")
print("="*76)


DATASET PREPARATION SUMMARY
Raw metadata rows         : 5,327
Raw images                : 5,328
Mapped metadata rows      : 5,326
Unmatched metadata        : 1
Unmatched images          : 2
Duplicate hash groups     : 33
Duplicate records removed : 33
Final modelling images    : 5,293

TARGET
Phase
Martensite    2823
Bainite       2470

SAMPLE-GROUPED SPLIT
Phase  Bainite  Martensite
split                     
test       342         418
train     1786        2025
val        342         380

MICROSCOPE HOLDOUTS
 heldout_microscope split  n_images  n_samples  bainite  martensite
                  0 train      2497         16     1138        1359
                  0   val       549          4      285         264
                  0  test      2247         20     1047        1200
                  1 train      3646         16     1657        1989
                  1   val       804          4      420         384
                  1  test       843         20      393         450
        

## Methodological interpretation

The preparation procedure establishes a cleaned, traceable cohort and several complementary evaluation regimes. The conventional split measures image-level predictive performance; the specimen-grouped split more directly probes transfer to new physical specimens. Microscope and magnification holdouts test acquisition-domain transfer under different constraints.

**Important qualifications:** exact-hash screening does not detect all visually related images, and the single-domain holdouts do not guarantee simultaneous specimen independence. The 100× domain also has a relatively small test cohort. Reported model performance should therefore always be associated with its specific evaluation protocol.

The exported manifests fix the dataset and partitions used for subsequent model comparisons.